In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# 1. Introduction

The purpose of this project is evaluate the performance of the animal image classification model

This was tested with 100 pictures of dogs and cats

In [4]:
data = pd.read_csv("animal-classifier-results.csv")
data.head()

,File name,Predicted category,Confidence,Actual category,Prediction correct?,Error,Dog score,Cat score,Bird score,Horse score,Squirrel score
0,Test 1/Cats_Test0.png,Cat,1.00000,Cat,True,NaN,5.224731e-23,1.000000e+00,4.759849e-29,2.484735e-35,1.475233e-21
1,Test 1/Cats_Test1.png,Dog,1.00000,Dog,True,NaN,1.000000e+00,0.000000e+00,1.937371e-34,2.044546e-27,0.000000e+00
2,Test 1/Cats_Test10.png,Dog,1.00000,Dog,True,NaN,1.000000e+00,1.140908e-31,1.097748e-32,4.865566e-21,6.825973e-33
3,Test 1/Cats_Test11.png,Cat,1.00000,Cat,True,NaN,5.338357e-38,1.000000e+00,0.000000e+00,4.393901e-35,3.974293e-30
4,Test 1/Cats_Test12.png,Dog,0.99999,Dog,True,NaN,9.999896e-01,9.736661e-06,1.912805e-22,1.111201e-14,6.228556e-07


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   File name            100 non-null    str    
 1   Predicted category   100 non-null    str    
 2   Confidence           100 non-null    float64
 3   Actual category      100 non-null    str    
 4   Prediction correct?  100 non-null    bool   
 5   Error                0 non-null      float64
 6   Dog score            100 non-null    float64
 7   Cat score            100 non-null    float64
 8   Bird score           100 non-null    float64
 9   Horse score          100 non-null    float64
 10  Squirrel score       100 non-null    float64
dtypes: bool(1), float64(7), str(3)
memory usage: 8.0 KB


In [6]:
data.describe()

,Confidence,Error,Dog score,Cat score,Bird score,Horse score,Squirrel score
count,100.000000,0.0,100.00000,1.000000e+02,1.000000e+02,1.000000e+02,1.000000e+02
mean,0.995374,NaN,0.72547,2.599520e-01,2.940540e-04,4.283528e-03,1.000002e-02
std,0.042888,NaN,0.44538,4.407621e-01,2.940539e-03,4.282096e-02,1.000000e-01
min,0.571788,NaN,0.00000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000,NaN,0.00136,4.189249e-35,1.467047e-34,1.133622e-34,1.007980e-36
50%,1.000000,NaN,1.00000,6.365165e-24,6.416146e-25,2.190152e-26,4.261748e-30
75%,1.000000,NaN,1.00000,9.973253e-01,2.219611e-19,3.204555e-20,5.547462e-23
max,1.000000,NaN,1.00000,1.000000e+00,2.940539e-02,4.282110e-01,1.000000e+00


# 2. Evaluating the relationship between the confidence of the answer and whether the machine was correct

In [8]:
df = pd.read_csv("animal-classifier-results.csv")
df.groupby("Prediction correct?")["Confidence"].describe()

,count,mean,std,min,25%,50%,75%,max
Prediction correct?,,,,,,,,
False,5.0,0.999406,0.001328,0.997031,1.0,1.0,1.0,1.0
True,95.0,0.995162,0.044003,0.571788,1.0,1.0,1.0,1.0


In [9]:
df["Confidence"].corr(df["Prediction correct?"].astype(int))

np.float64(-0.021678120072651796)

-0.0217 ≉  -1 => there is no significant relationship between confidence and a correct/ wrong answer

The standard deviation of correct predictions is noticeably larger than incorrect predictions, also the minimum is noticeably smaller. However, this seems to be an anomaly since the interquartile range is 0.
In order to test this hypothesis, further research must be drawn with a larger sample

# 3. Comparing the errors to identify relationships

In [19]:
errors = df[~df["Prediction correct?"]]
errors.round(3)

,File name,Predicted category,Confidence,Actual category,Prediction correct?,Error,Dog score,Cat score,Bird score,Horse score,Squirrel score
8,Test 1/Cats_Test16.png,Cat,0.997,Dog,False,NaN,0.003,0.997,0.0,0.0,0.0
53,Test 1/Cats_Test57.png,Cat,1.000,Dog,False,NaN,0.000,1.000,0.0,0.0,0.0
58,Test 1/Cats_Test61.png,Squirrel,1.000,Cat,False,NaN,0.000,0.000,0.0,0.0,1.0
73,Test 1/Cats_Test75.png,Dog,1.000,Cat,False,NaN,1.000,0.000,0.0,0.0,0.0
81,Test 1/Cats_Test82.png,Dog,1.000,Cat,False,NaN,1.000,0.000,0.0,0.0,0.0


In [21]:
def simplify(x):
    return x if x in ["Cat", "Dog"] else "Other"

actual = df["Actual category"].apply(simplify)
predicted = df["Predicted category"].apply(simplify)

table = pd.crosstab(actual, predicted)
table = table.reindex(index=["Cat", "Dog", "Other"], columns=["Cat", "Dog", "Other"], fill_value=0)
table.index.name = "Actual"
table.columns.name = "Predicted"
table

Predicted,Cat,Dog,Other
Actual,,,
Cat,24,2,1
Dog,2,71,0
Other,0,0,0


cats mistaken for dogs = 2
dogs mistaken for cats = 2
cats mistaken for others = 1

cats and dogs mistaken for eachother = 4
cats and dogs misaken for others = 1

In [27]:

table = pd.DataFrame(
    [[4, 2.5, 1.5, 2.25, 0.9],
     [1, 2.5, -1.5, 2.25, 0.9]],
    columns=["O", "E", "O-E", "(O-E)^2", "(O-E)^2/E"],
    index=["Cat and Dog", "Other"])

table

,O,E,O-E,(O-E)^2,(O-E)^2/E
Cat and Dog,4,2.5,1.5,2.25,0.9
Other,1,2.5,-1.5,2.25,0.9


df = 2-1 = 1 so critical value is 3.841
0.9 + 0.9 = 1.8
1.8 < 3.841
Therefore there is no statistical significant difference 

This test is inaccurate since we do not have enough values to test with. We should gather a larger sample with more errors in order to make this test


# 4 exploring the relationship between the models confidence and if the answer was correct 

In [34]:
df["Confidence band"] = pd.cut(df["Confidence"], [0, 0.9, 0.99, 0.999, 1.0])
df.groupby("Confidence band", observed=True)["Prediction correct?"].agg(["count", "mean"])

,count,mean
Confidence band,,
"(0.0, 0.9]",1,1.000000
"(0.9, 0.99]",1,1.000000
"(0.99, 0.999]",2,0.500000
"(0.999, 1.0]",96,0.958333


In [54]:
from scipy.stats import binomtest

k = df["Prediction correct?"].sum()
n = len(df)
avg_conf = df["Confidence"].mean()
print(f"Average confidence: {avg_conf:.5f}")
result = binomtest(k, n, avg_conf, alternative="less")
pvalue = result.pvalue
print(f"PValue: {pvalue:.5f}")

Average confidence: 0.99537
PValue: 0.00011


0.05 < 0.00011
Therefore, there is enough evidence to suggest, at 5% significance level, that the average confidence value is smaller than predicted. This means that the model is overconfident in its guesses which could be problematic as someone using the model would get no warning on the unreliability of the models prediction.

# 5 Conclusions and evaluations

This will be split into several sections:
5.1: Conclusions drawn from data analysis
5.2: Critisisms of analysis 
5.3: Suggestions for further improvement of the model

# 5.1 Conclusions drawn from data analysis

1. Unable to draw a significant relationship between confidence and a correct/ wrong answer
2. It seems there could be anomalies within the data which could be tested in future research
3. Unable to draw a statistical significant difference between cats and dogs being incorrectly confused between eachother compared to other animals
4. The model is overconfident

# 5.2 Critisisms of analysis

1. There was an underlying pattern showing that there was not enough data to draw reliable conclusions. Although this was meant to be a small pilot study, the data was insuficient. For example, there was only 5 errors to try and draw relationships with. 
Further research can be applied to fix this problem:
- A new pilot study with a larger data set
- A main study 
A new pilot study seems to be the way to go forward as the current study failed to draw enough reliable conclusions. This could save time as there may be drastic problems missed that can be fixed before a main study on the model.

2. Every image was labelled with "Cats_Test" even though some were images of dogs. This didn't provide any effect on the analysis or models accuracy however it is an organisation issue which should be addressed and fixed

3. The study only used dogs and cats in order to analyse the accuracy of the model. This leaves the unknown as to the models accuracy of the other animals. It should be assumed that the accuracy may vary as each category was trained with a different number of available images. Further research should be done to test this.

4. There could be more data extracted from each image. For example, the pilot study included some images of more 'unusual' looking animals which could have lead to a lower accuracy of the model. With more data on this, we could draw conclusions as to whether these images contributed significantly to the models innacuracy. This could give us ideas on future trainng to give the model.


# 5.2 Suggestions for further improvement of the model

1. Train the model on pictures of more 'unusual' looking animals within these categories
2. Train the model further with more data 
3. Fix the models overconfidence by training with label smoothing ensuring it isn't always fully certain which may lead to more errors